# Customer Support Intelligence Platform — Phase 3: Baseline ML Models

**Brief requirements covered (Day 7-8, plus Approach step 5):**
- Split data into train/validation/test sets using stratified splitting
- Baseline Models: Logistic Regression + Multinomial Naive Bayes on TF-IDF
  features (Ticket Type classification)
- Decision Tree baseline for Ticket Priority
- Evaluate using Accuracy, F1-macro, ROC-AUC
- Log all runs to MLflow

**Important correction from Phase 2:** the TF-IDF vectorizer built in the
preprocessing notebook was fit on the FULL dataset. That's a data leakage risk
- if the vectorizer learns vocabulary/weights from what will become our test
set, test performance would be artificially inflated (the model gets an
indirect preview of test data through the vectorizer's vocabulary). This
notebook refits TF-IDF properly, using ONLY the training split, before any
evaluation happens.

In [2]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import numpy as np
import joblib
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

RANDOM_STATE = 42  # fixed per brief's reproducibility requirement

df = pd.read_csv("../data_processed/tickets_with_features.csv")
print(f"Loaded {len(df)} rows")

Loaded 8469 rows


## 1. Train / Validation / Test split

**70/15/15 split, stratified on Ticket Type** (the "core NLP deliverable" per
the brief). Ticket Priority is fairly balanced independently (from our EDA),
so a single split stratified on Ticket Type keeps both targets' distributions
reasonably preserved without needing joint stratification, which sklearn
doesn't support directly for two separate label columns.

**This split is saved to disk and reused by every subsequent notebook** -
Days 9-10, 11, 12, and 13 all need to evaluate on the SAME test set for
results to be comparable.

In [3]:
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["Ticket Type"], random_state=RANDOM_STATE
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["Ticket Type"], random_state=RANDOM_STATE
)

print(f"Train: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
print(f"Val:   {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test:  {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")

import os
os.makedirs("../data_processed", exist_ok=True)
train_df.to_csv("../data_processed/train.csv", index=False)
val_df.to_csv("../data_processed/val.csv", index=False)
test_df.to_csv("../data_processed/test.csv", index=False)
print("\nSaved train/val/test splits - all later notebooks will load these exact files.")

print("\nTicket Type distribution check (should be similar % across all 3 splits):")
for name, split_df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"\n{name}:")
    print((split_df["Ticket Type"].value_counts(normalize=True) * 100).round(1))

Train: 5928 (70.0%)
Val:   1270 (15.0%)
Test:  1271 (15.0%)

Saved train/val/test splits - all later notebooks will load these exact files.

Ticket Type distribution check (should be similar % across all 3 splits):

Train:
Ticket Type
Refund request          20.7
Technical issue         20.6
Cancellation request    20.0
Product inquiry         19.4
Billing inquiry         19.3
Name: proportion, dtype: float64

Val:
Ticket Type
Refund request          20.7
Technical issue         20.6
Cancellation request    20.0
Product inquiry         19.4
Billing inquiry         19.3
Name: proportion, dtype: float64

Test:
Ticket Type
Refund request          20.7
Technical issue         20.6
Cancellation request    20.1
Product inquiry         19.4
Billing inquiry         19.3
Name: proportion, dtype: float64


## 2. Refit TF-IDF on training data only (leakage fix)

Refitting here, on `train_df` only, rather than reusing Phase 2's vectorizer
which saw the full dataset.

In [4]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(train_df["Combined_Text_Clean"])
X_val_tfidf = tfidf.transform(val_df["Combined_Text_Clean"])
X_test_tfidf = tfidf.transform(test_df["Combined_Text_Clean"])

print(f"TF-IDF shape - train: {X_train_tfidf.shape}, val: {X_val_tfidf.shape}, test: {X_test_tfidf.shape}")

joblib.dump(tfidf, "../models/tfidf_vectorizer_leakage_free.joblib")
print("Saved leakage-free vectorizer to models/tfidf_vectorizer_leakage_free.joblib")
print("(This is now the CORRECT one to use going forward - not Phase 2's version)")

TF-IDF shape - train: (5928, 5000), val: (1270, 5000), test: (1271, 5000)
Saved leakage-free vectorizer to models/tfidf_vectorizer_leakage_free.joblib
(This is now the CORRECT one to use going forward - not Phase 2's version)


## 3. MLflow setup

Tracking locally, as the brief specifies ("Configure MLflow tracking server
locally"). Every run logs hyperparameters, metrics, and the model artifact
itself, per the brief's explicit requirement.

In [5]:
mlflow.set_tracking_uri("../mlruns")
mlflow.set_experiment("customer_support_baselines")

def evaluate_classifier(y_true, y_pred, y_proba, label_names):
    """Standard evaluation used across all classification models in this
    project - keeps metrics consistent for fair comparison."""
    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average="macro")
    try:
        roc_auc = roc_auc_score(y_true, y_proba, multi_class="ovr", labels=label_names)
    except ValueError as e:
        roc_auc = None
        print(f"ROC-AUC could not be computed: {e}")
    return acc, f1_macro, roc_auc

2026/09/04 13:00:12 INFO mlflow.tracking.fluent: Experiment with name 'customer_support_baselines' does not exist. Creating a new experiment.


## 4. Baseline 1: Logistic Regression — Ticket Type (NLP)

In [6]:
y_train_type = train_df["Ticket Type"]
y_val_type = val_df["Ticket Type"]
type_labels = sorted(y_train_type.unique())

with mlflow.start_run(run_name="logreg_ticket_type_baseline"):
    lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    lr.fit(X_train_tfidf, y_train_type)

    y_pred = lr.predict(X_val_tfidf)
    y_proba = lr.predict_proba(X_val_tfidf)
    acc, f1_macro, roc_auc = evaluate_classifier(y_val_type, y_pred, y_proba, type_labels)

    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("random_state", RANDOM_STATE)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_macro", f1_macro)
    if roc_auc:
        mlflow.log_metric("roc_auc", roc_auc)
    mlflow.sklearn.log_model(lr, "model")

    print(f"Logistic Regression - Ticket Type")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  F1-macro: {f1_macro:.4f}")
    print(f"  ROC-AUC:  {roc_auc:.4f}" if roc_auc else "  ROC-AUC: N/A")
    print(f"\n{classification_report(y_val_type, y_pred)}")

2026/09/04 13:00:22 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the model.


Logistic Regression - Ticket Type
  Accuracy: 0.2071
  F1-macro: 0.2061
  ROC-AUC:  0.5061

                      precision    recall  f1-score   support

     Billing inquiry       0.21      0.19      0.20       245
Cancellation request       0.17      0.17      0.17       254
     Product inquiry       0.22      0.19      0.20       246
      Refund request       0.20      0.22      0.21       263
     Technical issue       0.23      0.26      0.24       262

            accuracy                           0.21      1270
           macro avg       0.21      0.21      0.21      1270
        weighted avg       0.21      0.21      0.21      1270



In [10]:
# DIAGNOSTIC: Sanity check - is this a code bug, or a genuine dataset property?
# ==============================================================================
# If a model performs the SAME on real labels as on deliberately SHUFFLED
# (scrambled) labels, that's definitive proof the real labels carry no more
# signal than random noise - confirming this is a property of the dataset,
# not a bug in our train/test split, encoding, or TF-IDF pipeline.

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

np.random.seed(42)
y_train_shuffled = np.random.permutation(y_train_type.values)

lr_shuffled = LogisticRegression(max_iter=1000, random_state=42)
lr_shuffled.fit(X_train_tfidf, y_train_shuffled)
y_pred_shuffled = lr_shuffled.predict(X_val_tfidf)
acc_shuffled = accuracy_score(y_val_type, y_pred_shuffled)

print(f"Logistic Regression accuracy on REAL labels:     {acc:.4f}")
print(f"Logistic Regression accuracy on SHUFFLED labels: {acc_shuffled:.4f}")
print()
if abs(acc - acc_shuffled) < 0.03:
    print("CONFIRMED: performance on real labels is statistically indistinguishable")
    print("from performance on randomly shuffled labels. This is definitive evidence")
    print("that Ticket Type has no learnable relationship to the ticket text in this")
    print("dataset - a genuine property of this specific data, not a bug in our code.")
else:
    print("Real labels perform meaningfully better than shuffled labels - there IS")
    print("some learnable signal, even if weak. Worth investigating why baseline")
    print("models aren't capturing more of it.")

Logistic Regression accuracy on REAL labels:     0.2717
Logistic Regression accuracy on SHUFFLED labels: 0.1709

Real labels perform meaningfully better than shuffled labels - there IS
some learnable signal, even if weak. Worth investigating why baseline
models aren't capturing more of it.


## 5. Baseline 2: Multinomial Naive Bayes — Ticket Type (NLP)

In [7]:
with mlflow.start_run(run_name="naive_bayes_ticket_type_baseline"):
    nb_model = MultinomialNB()
    nb_model.fit(X_train_tfidf, y_train_type)

    y_pred = nb_model.predict(X_val_tfidf)
    y_proba = nb_model.predict_proba(X_val_tfidf)
    acc, f1_macro, roc_auc = evaluate_classifier(y_val_type, y_pred, y_proba, type_labels)

    mlflow.log_param("model_type", "MultinomialNB")
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_macro", f1_macro)
    if roc_auc:
        mlflow.log_metric("roc_auc", roc_auc)
    mlflow.sklearn.log_model(nb_model, "model")

    print(f"Multinomial Naive Bayes - Ticket Type")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  F1-macro: {f1_macro:.4f}")
    print(f"  ROC-AUC:  {roc_auc:.4f}" if roc_auc else "  ROC-AUC: N/A")
    print(f"\n{classification_report(y_val_type, y_pred)}")

2026/09/04 13:00:25 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the model.


Multinomial Naive Bayes - Ticket Type
  Accuracy: 0.2134
  F1-macro: 0.2090
  ROC-AUC:  0.5108

                      precision    recall  f1-score   support

     Billing inquiry       0.20      0.15      0.17       245
Cancellation request       0.20      0.20      0.20       254
     Product inquiry       0.22      0.16      0.18       246
      Refund request       0.21      0.25      0.23       263
     Technical issue       0.23      0.30      0.26       262

            accuracy                           0.21      1270
           macro avg       0.21      0.21      0.21      1270
        weighted avg       0.21      0.21      0.21      1270



## 6. Baseline 3: Decision Tree — Ticket Priority (Tabular)

Priority prediction uses structured/tabular features (age, encoded
categoricals, tenure, text stats) rather than TF-IDF - matches the brief's
description of this as a separate "Tabular ML" task from Ticket Type's NLP task.

In [8]:
TABULAR_FEATURES = [
    "Customer Age", "Ticket Channel_Encoded", "Product Purchased_Encoded",
    "Customer Gender_Encoded", "Customer_Tenure_Days",
    "Description_Char_Count", "Description_Word_Count", "Sentiment_Polarity",
]

X_train_tab = train_df[TABULAR_FEATURES]
X_val_tab = val_df[TABULAR_FEATURES]
y_train_priority = train_df["Ticket Priority"]
y_val_priority = val_df["Ticket Priority"]
priority_labels = sorted(y_train_priority.unique())

with mlflow.start_run(run_name="decision_tree_ticket_priority_baseline"):
    dt = DecisionTreeClassifier(max_depth=10, random_state=RANDOM_STATE)
    dt.fit(X_train_tab, y_train_priority)

    y_pred = dt.predict(X_val_tab)
    y_proba = dt.predict_proba(X_val_tab)
    acc, f1_macro, roc_auc = evaluate_classifier(y_val_priority, y_pred, y_proba, priority_labels)

    mlflow.log_param("model_type", "DecisionTreeClassifier")
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("random_state", RANDOM_STATE)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_macro", f1_macro)
    if roc_auc:
        mlflow.log_metric("roc_auc", roc_auc)
    mlflow.sklearn.log_model(dt, "model")

    print(f"Decision Tree - Ticket Priority")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  F1-macro: {f1_macro:.4f}")
    print(f"  ROC-AUC:  {roc_auc:.4f}" if roc_auc else "  ROC-AUC: N/A")
    print(f"\n{classification_report(y_val_priority, y_pred)}")

2026/09/04 13:00:28 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the model.


Decision Tree - Ticket Priority
  Accuracy: 0.2717
  F1-macro: 0.2667
  ROC-AUC:  0.5146

              precision    recall  f1-score   support

    Critical       0.28      0.29      0.28       336
        High       0.29      0.21      0.25       307
         Low       0.24      0.21      0.22       292
      Medium       0.27      0.36      0.31       335

    accuracy                           0.27      1270
   macro avg       0.27      0.27      0.27      1270
weighted avg       0.27      0.27      0.27      1270



## 7. Baseline summary

These 3 numbers are our floor - every subsequent, more sophisticated model
(Random Forest, XGBoost, BiLSTM, DistilBERT) should beat these, or we'd have
no justification for the added complexity.

In [9]:
print("BASELINE RESULTS SUMMARY (validation set)")
print("=" * 60)
print("Run: mlflow ui --backend-store-uri ../mlruns")
print("to view all logged runs in the MLflow web dashboard.")
print()
print("Ticket Type (NLP task):")
print("  Logistic Regression and Multinomial Naive Bayes results printed above")
print()
print("Ticket Priority (Tabular task):")
print("  Decision Tree results printed above")
print()
print("All 3 runs logged to MLflow experiment 'customer_support_baselines'")

BASELINE RESULTS SUMMARY (validation set)
Run: mlflow ui --backend-store-uri ../mlruns
to view all logged runs in the MLflow web dashboard.

Ticket Type (NLP task):
  Logistic Regression and Multinomial Naive Bayes results printed above

Ticket Priority (Tabular task):
  Decision Tree results printed above

All 3 runs logged to MLflow experiment 'customer_support_baselines'
